# Módulo 45 · DL I: redes neuronales con Keras (cáncer de mama)

Código del módulo para ejecutar en **Google Colab**, donde TensorFlow y PyTorch ya vienen instalados. Para entrenar más rápido activa la GPU en *Entorno de ejecución → Cambiar tipo de entorno de ejecución*. Las explicaciones, los ejercicios y las partes que funcionan en el navegador están en el módulo 45 de la web.

In [ ]:
# Solo si lo ejecutas en tu ordenador (en Colab ya está todo instalado):
# !pip install tensorflow scikit-learn matplotlib

### Keras / TensorFlow (Colab: cuaderno del módulo)

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt

tf.keras.utils.set_random_seed(42)            # fija Python, NumPy y TensorFlow (np.random.seed solo fija NumPy)

data = load_breast_cancer()
X_train, X_test, y_train, y_test = train_test_split(data.data, data.target, test_size=0.2,
                                                    random_state=42, stratify=data.target)
scaler = StandardScaler().fit(X_train)        # primero separar, después escalar
X_train, X_test = scaler.transform(X_train), scaler.transform(X_test)

model = models.Sequential([
    layers.Input(shape=(X_train.shape[1],)),  # forma recomendada en Keras 3 (en vez de input_shape=)
    layers.Dense(30, activation="relu"),
    layers.Dense(50, activation="relu"),
    layers.Dense(10, activation="relu"),
    layers.Dense(1, activation="sigmoid"),    # probabilidad de la clase 1 (benigno)
])
model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
model.summary()

parada = tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=True)
history = model.fit(X_train, y_train, epochs=100, batch_size=16, validation_split=0.2,
                    callbacks=[parada], verbose=0)
print("Épocas entrenadas:", len(history.history["loss"]))

fig, axs = plt.subplots(1, 2, figsize=(12, 4))
for ax, m in zip(axs, ["loss", "accuracy"]):
    ax.plot(history.history[m], label="entrenamiento")
    ax.plot(history.history["val_" + m], label="validación")
    ax.set_title(m); ax.set_xlabel("Época"); ax.legend()
plt.show()

y_pred = (model.predict(X_test, verbose=0) > 0.5).astype(int).ravel()
print(classification_report(y_test, y_pred, target_names=data.target_names))
print(confusion_matrix(y_test, y_pred))